In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 检查有没有 GPU，没有就用 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 把图片转成 Tensor（张量），并归一化
transform = transforms.Compose([transforms.ToTensor()])

# 下载并加载 训练集 和 测试集
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# DataLoader 负责把数据“分批次(Batch)”喂给模型
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        # 输入是 28*28=784 个像素，第一层输出 128 个神经元
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()          
        self.fc2 = nn.Linear(128, 10)  

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x 

model = SimpleNet().to(device)

#交叉熵损失
criterion = nn.CrossEntropyLoss() 
#SGD负责根据梯度调整模型参数
optimizer = optim.SGD(model.parameters(), lr=0.01)

model.train() # 切换到训练模式
for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)

    optimizer.zero_grad()

    output = model(data)

    loss = criterion(output, target)

    loss.backward()

    optimizer.step()
    
    if batch_idx % 100 == 0:
        print(f"训练进度: {batch_idx * len(data)}/{len(train_loader.dataset)} Loss: {loss.item():.4f}")